# 01 — Exploration du dataset
> Analyse exploratoire des images histologiques du cancer du sein.
> Objectif : comprendre la distribution des classes, la qualité et la diversité des images.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import Counter
from PIL import Image

# Reproductibilité
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ✅ MODIFIÉ : TRAIN_DIR au lieu de RAW_DIR (dataset déjà splitté par le prof)
from config import TRAIN_DIR, CLASS_NAMES
print(f"Dataset path : {TRAIN_DIR.resolve()}")
print(f"Classes attendues : {CLASS_NAMES}")

## 1. Distribution des classes

In [ ]:
# Comptage des images par classe dans le set train
counts = {}
# ✅ MODIFIÉ : TRAIN_DIR au lieu de RAW_DIR
for cls_dir in sorted(TRAIN_DIR.iterdir()):
    if cls_dir.is_dir():
        n = len([f for f in cls_dir.iterdir()
                 if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}])
        counts[cls_dir.name] = n

total = sum(counts.values())
print(f"Total images (train) : {total}")
for cls, n in counts.items():
    print(f"  {cls:>12} : {n:>5} images ({100*n/total:.1f}%)")

# Vérification déséquilibre
if len(counts) == 2:
    vals = list(counts.values())
    ratio = max(vals) / min(vals)
    print(f"\nRatio déséquilibre : {ratio:.2f}:1")
    if ratio > 1.5:
        print("  ⚠ Déséquilibre détecté → on utilisera class_weight dans la Loss")
    else:
        print("  ✓ Classes relativement équilibrées")

In [ ]:
# Visualisation de la distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#1D9E75', '#E24B4A']
bars = axes[0].bar(counts.keys(), counts.values(),
                   color=colors[:len(counts)], edgecolor='white', width=0.5)
axes[0].set_title("Nombre d'images par classe (train)", fontsize=12)
axes[0].set_ylabel("Nombre d'images")
for bar, (cls, n) in zip(bars, counts.items()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(n), ha='center', va='bottom', fontweight='bold')

axes[1].pie(counts.values(), labels=counts.keys(), colors=colors[:len(counts)],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proportion des classes', fontsize=12)

plt.suptitle('Distribution du dataset (split train)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../models/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Visualisation d'un échantillon par classe

In [ ]:
fig, axes = plt.subplots(len(counts), 5, figsize=(16, 4 * len(counts)))
if len(counts) == 1:
    axes = [axes]

for row, (cls_name, _) in enumerate(counts.items()):
    # ✅ MODIFIÉ : TRAIN_DIR au lieu de RAW_DIR
    cls_dir = TRAIN_DIR / cls_name
    images = list(cls_dir.iterdir())
    sample = random.sample(images, min(5, len(images)))

    for col, img_path in enumerate(sample):
        img = Image.open(img_path).convert('RGB')
        axes[row][col].imshow(img)
        axes[row][col].set_title(f'{cls_name}\n{img.size[0]}x{img.size[1]}', fontsize=8)
        axes[row][col].axis('off')

plt.suptitle("Échantillon d'images par classe (train)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3. Analyse des dimensions d'images

In [ ]:
dims = {}
# ✅ MODIFIÉ : TRAIN_DIR au lieu de RAW_DIR
for cls_dir in sorted(TRAIN_DIR.iterdir()):
    if not cls_dir.is_dir():
        continue
    cls_dims = []
    for f in cls_dir.iterdir():
        if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}:
            try:
                w, h = Image.open(f).size
                cls_dims.append((w, h))
            except Exception:
                pass
    dims[cls_dir.name] = cls_dims

print("Statistiques des dimensions (largeur x hauteur) :")
for cls, d in dims.items():
    if not d:
        continue
    ws, hs = zip(*d)
    print(f"\n{cls}:")
    print(f"  Largeur  : min={min(ws)} max={max(ws)} moy={np.mean(ws):.0f} std={np.std(ws):.0f}")
    print(f"  Hauteur  : min={min(hs)} max={max(hs)} moy={np.mean(hs):.0f} std={np.std(hs):.0f}")
print("\n→ Resize à 224x224 nécessaire pour EfficientNet-B3")

## 4. Détection d'images corrompues

In [ ]:
from PIL import UnidentifiedImageError

corrupted = []
# ✅ MODIFIÉ : TRAIN_DIR au lieu de RAW_DIR
for cls_dir in TRAIN_DIR.iterdir():
    if not cls_dir.is_dir():
        continue
    for f in cls_dir.iterdir():
        if f.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp'}:
            continue
        try:
            with Image.open(f) as img:
                img.verify()
        except (UnidentifiedImageError, Exception) as e:
            corrupted.append((f, str(e)))

print(f"Images corrompues détectées : {len(corrupted)}")
for path, err in corrupted[:10]:
    print(f"  {path.name}: {err}")
if not corrupted:
    print("✓ Aucune image corrompue — dataset propre")

## 5. Conclusion

Complète avec tes observations après exécution :

In [ ]:
print("=" * 50)
print("RÉSUMÉ EXPLORATION")
print("=" * 50)
print(f"Total images (train) : {total}")
for cls, n in counts.items():
    print(f"{cls:>15} : {n} images ({100*n/total:.1f}%)")
print(f"Images corrompues    : {len(corrupted)}")
print("")
print("Actions à mener :")
print("  1. Resize toutes les images à 224x224")
print("  2. Appliquer augmentation sur le train set")
print("  3. Utiliser class_weight si déséquilibre > 1.5:1")
print("  4. Supprimer les images corrompues si nécessaire")